# Load library

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as skl
import anndata as ann
import random, os
from scipy.stats import pearsonr as pr
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score as f1
from sklearn.metrics import precision_recall_curve as prc
from sklearn.metrics import silhouette_score as sil
from sklearn.metrics import auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, average_precision_score
from sklearn.metrics import silhouette_score
from torch_geometric.nn import TransformerConv
from torch_geometric.data import Data
import psutil
import os, sys
import gc
import scipy.sparse as sp
from harmony import harmonize
from tqdm import tqdm
import h5py

In [2]:
sc.set_figure_params(dpi=200)

# General processing functinos

In [3]:
def whats_memory_eater():
    # Build reverse map of object id -> variable name from globals
    name_map = {id(obj): name for name, obj in globals().items()}

    # Get all tracked objects
    all_objects = gc.get_objects()

    # Safely get size and match variable name
    sizes = []
    for obj in all_objects:
        try:
            size = sys.getsizeof(obj)
            obj_id = id(obj)
            name = name_map.get(obj_id, None)
            sizes.append((size, type(obj), name, repr(obj)[:100]))
        except Exception:
            continue

    # Sort and print top 10
    sizes.sort(reverse=True, key=lambda x: x[0])

    for size, obj_type, name, preview in sizes[:10]:
        print(f"Size: {size / 1024**3} GB | Type: {obj_type} | Name: {name} | Object: {preview}")


In [4]:
def memory_usgae():
    gc.collect()
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / 1024**3  # in GB

    print(f"Current memory usage: {memory_gb:.2f} GB")

In [5]:
def load_and_preprocess_project(base_path, Project_ID, metadata_idx_key='Cell', 
                                Primary_or_Metastatic = 'Primary', remove_doublets = True, doublet_rate=0.06,
                                further_pre = False, file_prefix= None):
    """
    Load and preprocess a single scRNA-seq project with standard filtering and UMAP.
    
    Assumes the base_path contains:
        - One .mtx file (count matrix)
        - One barcodes.csv
        - One features.csv
        - One meta_all.csv
    """
    # Automatically detect files
    files = os.listdir(base_path)
    metadata_file = None
    
    if file_prefix == None:

        mtx_file = [os.path.join(base_path, f) for f in files if f.endswith('.mtx')][0]
        print(mtx_file)
        barcodes_file = [os.path.join(base_path, f) for f in files if 'barcode' in f][0]
        print(barcodes_file)
        try:
            features_file = [os.path.join(base_path, f) for f in files if 'feature' in f][0]
        except:
            features_file = [os.path.join(base_path, f) for f in files if 'genes' in f][0]
        print(features_file)
        metadata_file = [os.path.join(base_path, f) for f in files if 'meta' in f][0]
        print(metadata_file)
    else:
        for f in files:
            if not f.startswith(file_prefix):
                continue
            if f.endswith('mtx'):
                mtx_file = os.path.join(base_path, f)
                print(mtx_file)
            elif 'barcode' in f:
                barcodes_file = os.path.join(base_path, f)
                print(barcodes_file)
            elif 'feature' in f:
                features_file = os.path.join(base_path, f)
                print(features_file)
            elif 'genes' in f:
                features_file = os.path.join(base_path, f)
                print(features_file)
            elif 'meta' in f:
                metadata_file = os.path.join(base_path, f)
                print(metadata_file)
            else:
                continue
    # print(metadata_file)
    print(f"Loading: {mtx_file}")

    # Load matrix
    adata = sc.read_mtx(mtx_file)
    adata = adata.transpose()  # Important: make cells as rows, genes as columns

    # Load barcodes and features
    if barcodes_file.endswith('tsv'):
        barcodes = pd.read_csv(barcodes_file, sep='\t', header=None)  # no header=None here
    else:
        barcodes = pd.read_csv(barcodes_file)  # no header=None here
    display(barcodes)
    
    if features_file.endswith('tsv'):
        genes = pd.read_csv(features_file, sep='\t', header=None)  # no header=None here
    else:
        genes = pd.read_csv(features_file)  # no header=None here
    display(genes)

    # Assign barcodes and gene names (convert to string)
    if barcodes.shape[1] > 1:
        adata.obs_names = barcodes.iloc[:, 1].astype(str).values
    else:
        adata.obs_names = barcodes.iloc[:, 0].astype(str).values
    
    if genes.shape[1] > 1:
        adata.var_names = genes.iloc[:, 1].astype(str).values
    else:
        adata.var_names = genes.iloc[:, 0].astype(str).values
    # adata.var_names = genes.iloc[:, 0].astype(str).values
    display(adata.to_df())

    # Load and merge metadata
    # if metadata_file
    try:
        if metadata_file.endswith('tsv'):
            metadata = pd.read_csv(metadata_file, sep='\t')
        elif metadata_file.endswith('csv'):
            metadata = pd.read_csv(metadata_file)
        metadata.index = metadata[metadata_idx_key]
        adata.obs = adata.obs.join(metadata, how='left')
    except:
        pass         
    
    # DOUBLET DETECTION (before other filtering)
    if remove_doublets:
        print(f'Running doublet detection on {adata.n_obs} cells...')
                
        sc.external.pp.scrublet(adata, expected_doublet_rate=0.06)
        
        n_cells = adata.n_obs
        n_doublets = adata.obs['predicted_doublet'].sum()
        print(f'  Detected {n_doublets} doublets ({n_doublets/n_cells*100:.1f}%)')
        
        adata = adata[~adata.obs['predicted_doublet']].copy()
        print(f'  After doublet removal: {adata.n_obs} cells')

    # Calculate QC metrics
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

    # Standard cell filtering
    adata = adata[(adata.obs['n_genes_by_counts'] >= 200) & 
                  (adata.obs['n_genes_by_counts'] <= 5000) & 
                  (adata.obs['pct_counts_mt'] <= 20)].copy()
    
    adata.obs['Project_ID'] = Project_ID
    adata.obs['Primary_or_Metastatic'] = Primary_or_Metastatic
    
    # Normalize and log transform
    adata.raw = adata.copy()
    
    if further_pre:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

        # Highly variable genes
        # sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

        # Keep only HVGs
        # adata = adata[:, adata.var.highly_variable]

        # Scale
        sc.pp.scale(adata, max_value=10)

        # PCA
        sc.tl.pca(adata, svd_solver='arpack')

        # Neighbors
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)

        # UMAP
        sc.tl.umap(adata)

    print(f"Finished processing {Project_ID}. Shape: {adata.shape}")

    return adata


In [6]:
def filter_and_recompute(adata, celltype_col, celltypes_to_keep, further_pre = False):
    """
    Filters an AnnData object to keep only specified cell types, 
    then recalculates PCA, neighbors, and UMAP.

    Parameters:
    - adata: AnnData object
    - celltype_col: str, the column in adata.obs containing cell type annotations
    - celltypes_to_keep: list of str, the cell types you want to keep

    Returns:
    - filtered and recalculated AnnData object
    """
    # Step 1: Filter cells
    print(f"Original shape: {adata.shape}")
    adata_filtered = adata[adata.obs[celltype_col].isin(set(celltypes_to_keep))].copy()
    print(f"Filtered shape: {adata_filtered.shape}")

    # Step 2: Recalculate PCA and UMAP
    # (Assumes data is already normalized and scaled)
    if further_pre:
        sc.tl.pca(adata_filtered, svd_solver='arpack')
        sc.pp.neighbors(adata_filtered, n_neighbors=15, n_pcs=40)
        sc.tl.umap(adata_filtered)

    print("Recalculated PCA and UMAP.")
    return adata_filtered

In [7]:
def reprocess_from_raw_layer(adata, Project_ID, Primary_or_Metastatic='Primary', 
                             further_pre=False, remove_doublets=True, doublet_rate=0.06):
    """
    Reprocess a Scanpy AnnData object using its raw layer (e.g., from a published .h5ad).
    This includes doublet removal, normalization, HVG selection, PCA, neighbors, and UMAP.
    
    Parameters:
    - adata: AnnData object, must have .raw set
    - Project_ID: Project identifier
    - Primary_or_Metastatic: Sample type ('Primary' or 'Metastatic')
    - further_pre: Whether to do full preprocessing (normalization, PCA, UMAP)
    - remove_doublets: Whether to run doublet detection and filtering
    - doublet_rate: Expected doublet rate (default 0.06 = 6%)
    
    Returns:
    - Processed AnnData object (modifies in place)
    """
    # Check if raw exists
    if adata.raw is None:
        raise ValueError("AnnData object has no .raw attribute. Cannot proceed with reprocessing.")
    
    # Extract raw counts
    adata.X = adata.raw.X.copy()
    adata.var = adata.raw.var.copy()
    adata.var_names = adata.raw.var_names.copy()
    
    # DOUBLET DETECTION (before other filtering)
    if remove_doublets:
        print(f'Running doublet detection on {adata.n_obs} cells...')
                
        sc.external.pp.scrublet(adata, expected_doublet_rate=0.06)
        
        n_cells = adata.n_obs
        n_doublets = adata.obs['predicted_doublet'].sum()
        print(f'  Detected {n_doublets} doublets ({n_doublets/n_cells*100:.1f}%)')
        
        adata = adata[~adata.obs['predicted_doublet']].copy()
        print(f'  After doublet removal: {adata.n_obs} cells')
    
    # Recalculate mitochondrial content
    adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    
    print('Standard filtering...')
    # Standard filtering (optional)
    adata = adata[(adata.obs['n_genes_by_counts'] >= 200) &
                  (adata.obs['n_genes_by_counts'] <= 5000) &
                  (adata.obs['pct_counts_mt'] <= 20)].copy()
    
    # Add metadata
    adata.obs['Project_ID'] = Project_ID
    adata.obs['Primary_or_Metastatic'] = Primary_or_Metastatic
    
    if further_pre:
        print('Normalizing...')
        # Normalize and log transform
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        
        print('Scaling...')
        sc.pp.scale(adata, max_value=10)
        
        print('Computing PCA...')
        sc.tl.pca(adata, svd_solver='arpack')
        
        print('Computing neighbors...')
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
        
        print('Computing UMAP...')
        sc.tl.umap(adata)
    
    print(f"Reprocessed dataset. Final shape: {adata.shape}")
    return adata

In [8]:
def reprocess_all(adata, further_pre = True):
    """
    Reprocess a Scanpy AnnData object using its raw layer (e.g., from a published .h5ad).
    This includes normalization, HVG selection, PCA, neighbors, and UMAP.

    Parameters:
    - adata: AnnData object, must have .raw set

    Returns:
    - Processed AnnData object (modifies in place)
    """

    # Check if raw exists
    if adata.raw is None:
        raise ValueError("AnnData object has no .raw attribute. Cannot proceed with reprocessing.")

    # Extract raw counts
    adata.X = adata.raw.X.copy()
    adata.var = adata.raw.var.copy()
    adata.var_names = adata.raw.var_names.copy()

    # Recalculate mitochondrial content
    adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    
    print('Standard filtering...')
    # Standard filtering (optional)
    adata = adata[(adata.obs['n_genes_by_counts'] >= 200) &
                  (adata.obs['n_genes_by_counts'] <= 5000) &
                  (adata.obs['pct_counts_mt'] <= 20)].copy()
    
    # adata.obs['Project_ID'] = Project_ID
    # adata.obs['Primary_or_Metastatic'] = Primary_or_Metastatic
    
    if further_pre:
        # Normalize and log transform
        print('Normalizing...')
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

        # HVG selection
        # sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
        # adata = adata[:, adata.var.highly_variable]
        '''
        sc.pp.highly_variable_genes(
            adata,
            flavor="seurat_v3",  # best for batch-aware HVG selection
            n_top_genes=2000,
            batch_key="Final_sample_id"  # or whatever your batch label column is
        )
        '''
        
        # adata = adata[:, adata.var.highly_variable].copy()

        # Scale
        # print('Scaling...')
        # sc.pp.scale(adata, max_value=10)
        print('Computing PCA...')
        # PCA, neighbors, UMAP
        sc.tl.pca(adata, zero_center=False)
        
        print('Computing neighbors...')
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
        
        print('Computing UMAP...')
        sc.tl.umap(adata)

    print(f"Reprocessed dataset. Final shape: {adata.shape}")
    return adata


In [9]:
def reset_plot():
    # After running scrublet, reset the display settings
    import matplotlib.pyplot as plt
    import matplotlib
    %matplotlib inline

    # Reset matplotlib backend
    matplotlib.use('module://matplotlib_inline.backend_inline')

    # Reset scanpy settings
    import scanpy as sc
    sc.settings.autoshow = True
    sc.settings.set_figure_params(dpi=100, facecolor='white')

# Colorectal Cancer (COAD)

## A pan-cancer blueprint of the heterogeneous tumor microenvironment revealed by single-cell profiling


Paper: https://www.nature.com/articles/s41422-020-0355-0#Fig3

Data downloaded from: https://lambrechtslab.sites.vib.be/en/pan-cancer-blueprint-tumour-microenvironment-0

Link: 
- Matrix: https://lambrechtslab.sites.vib.be/en/pan-cancer-blueprint-tumour-microenvironment-0 (Colorectal cancer - Counts Matrix)
- Patient metadata: https://static-content.springer.com/esm/art%3A10.1038%2Fs41422-020-0355-0/MediaObjects/41422_2020_355_MOESM13_ESM.pdf
- Sequecing quality: https://www.nature.com/
https://static-content.springer.com/esm/art%3A10.1038%2Fs41422-020-0355-0/MediaObjects/41422_2020_355_MOESM14_ESM.pdf

In [10]:
memory_usgae()

Current memory usage: 0.65 GB


In [11]:
# Set the project directory
project_dir = "../../Data/COAD/2098-Colorectalcancer/"  # <- change this for each project

# Load and preprocess
ad = load_and_preprocess_project(project_dir, 
                                 Project_ID='2098-Colorectalcancer', 
                                 Primary_or_Metastatic='Primary',
                                 further_pre=True)

../../Data/COAD/2098-Colorectalcancer/matrix.mtx
../../Data/COAD/2098-Colorectalcancer/barcodes.tsv
../../Data/COAD/2098-Colorectalcancer/genes.tsv
../../Data/COAD/2098-Colorectalcancer/2099-Colorectalcancer_metadata.csv
Loading: ../../Data/COAD/2098-Colorectalcancer/matrix.mtx


,0
0,scrEXT001_AAACCTGGTCGGCTCA
1,scrEXT001_AAACCTGGTCTTTCAT
2,scrEXT001_AAACCTGTCACCACCT
3,scrEXT001_AAACCTGTCGTCCAGG
4,scrEXT001_AAACCTGTCTGGGCCA
...,...
44679,scrEXT029_TTTGTCACAGGGCATA
44680,scrEXT029_TTTGTCAGTAATCACC
44681,scrEXT029_TTTGTCAGTCTAGCCG
44682,scrEXT029_TTTGTCAGTTAGATGA


,0,1
0,RP11-34P13.3,RP11-34P13.3
1,FAM138A,FAM138A
2,OR4F5,OR4F5
3,RP11-34P13.7,RP11-34P13.7
4,RP11-34P13.8,RP11-34P13.8
...,...,...
33689,AC233755.2,AC233755.2
33690,AC233755.1,AC233755.1
33691,AC240274.1,AC240274.1
33692,AC213203.1,AC213203.1


,RP11-34P13.3,FAM138A,OR4F5,RP11-34P13.7,RP11-34P13.8,RP11-34P13.14,RP11-34P13.9,FO538757.3,FO538757.2,AP006222.2,...,AC007325.2,BX072566.1,AL354822.1,AC023491.2,AC004556.1,AC233755.2,AC233755.1,AC240274.1,AC213203.1,FAM231B
scrEXT001_AAACCTGGTCGGCTCA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
scrEXT001_AAACCTGGTCTTTCAT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
scrEXT001_AAACCTGTCACCACCT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
scrEXT001_AAACCTGTCGTCCAGG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
scrEXT001_AAACCTGTCTGGGCCA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
scrEXT029_TTTGTCACAGGGCATA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
scrEXT029_TTTGTCAGTAATCACC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
scrEXT029_TTTGTCAGTCTAGCCG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
scrEXT029_TTTGTCAGTTAGATGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Running doublet detection on 44684 cells...


/home/wang3712/.local/lib/python3.8/site-packages/scanpy/preprocessing/_normalization.py:170: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


Automatically set threshold at doublet score = 0.82
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 0.1%
Overall doublet rate:
	Expected   = 6.0%
	Estimated  = 0.0%
  Detected 0 doublets (0.0%)
  After doublet removal: 44684 cells


/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1086: NumbaDeprecationWarning:

Finished processing 2098-Colorectalcancer. Shape: (39963, 33694)


In [12]:
ad.obs['PatientNumber'] = ad.obs['PatientNumber'].astype(str)

In [13]:
ad = filter_and_recompute(adata=ad, 
                          celltype_col='CellType', 
                          celltypes_to_keep=['Cancer'],
                          further_pre=True)
ad

Original shape: (39963, 33694)
Filtered shape: (9039, 33694)
Recalculated PCA and UMAP.


AnnData object with n_obs × n_vars = 9039 × 33694
    obs: 'Cell', 'nGene', 'nUMI', 'CellFromTumor', 'PatientNumber', 'TumorType', 'TumorSite', 'CellType', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    uns: 'scrublet', 'log1p', 'pca', 'neighbors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [15]:
patient_cell_number = pd.read_csv("../../Data/COAD/2098-Colorectalcancer/2099-Colorectalcancer_metadata.csv")['PatientNumber'].value_counts()
patient_cell_number = patient_cell_number.to_dict()
patient_cell_number

{32: 11875, 31: 8279, 35: 8038, 33: 7098, 38: 4434, 36: 2591, 37: 2369}

In [16]:
pd.read_csv('../../Data/BRCA/2102-Breastcancer/Sequencing_quality_S2.txt', sep='\t')['Cancer type'].value_counts()

Cancer type
LC     36
CRC    21
BC     19
OvC    10
Name: count, dtype: int64

In [17]:
patient_seuqncing_meta_df = pd.read_csv('../../Data/BRCA/2102-Breastcancer/Sequencing_quality_S2.txt', sep='\t')
patient_seuqncing_meta_df = patient_seuqncing_meta_df[patient_seuqncing_meta_df['Cancer type'] == 'CRC']
patient_seuqncing_meta_df

,Patient number,Cancer type,10X version,Cells,Sample type,Tumour site,UMIs,Saturation (%),Reads
46,CRC_1,CRC,3' V2,2712,Tumour,Core,23040771,74.1,267375232
47,CRC_1,CRC,3' V2,2901,Tumour,Border,26281216,72.2,275655234
48,CRC_1,CRC,3' V2,2666,Normal,na,22077384,67.6,161020595
49,CRC_2,CRC,3' V2,4936,Tumour,Core,23780187,76.5,266870550
50,CRC_2,CRC,3' V2,4461,Tumour,Border,20942817,79.5,255400094
51,CRC_2,CRC,3' V2,2478,Normal,na,20498399,67.8,149987327
52,CRC_3,CRC,3' V2,2871,Tumour,Core,21895911,68.5,173029484
53,CRC_3,CRC,3' V2,2323,Tumour,Border,16041374,72.5,148821159
54,CRC_3,CRC,3' V2,1904,Normal,na,8600112,77.6,126457430
55,CRC_4,CRC,3' V2,3770,Tumour,Core,16529472,55.0,200491544


In [18]:
# Group by 'Patient number' and sum the 'Cells' column
cells_to_lc_label = patient_seuqncing_meta_df.groupby("Patient number")["Cells"].sum().to_dict()

# Flip the dict so it's {cell_sum: patient_id}
cells_to_lc_label = {v: k for k, v in cells_to_lc_label.items()}
cells_to_lc_label


{8279: 'CRC_1',
 11875: 'CRC_2',
 7098: 'CRC_3',
 8038: 'CRC_4',
 2591: 'CRC_5',
 2369: 'CRC_6',
 4434: 'CRC_7'}

In [19]:
patient_number_to_LC_id = dict()
for patient_number in patient_cell_number.keys():
    # print(patient_number)
    try:
        patient_number_to_LC_id[str(patient_number)] = cells_to_lc_label[patient_cell_number[patient_number]]
    except:
        print(patient_number)
patient_number_to_LC_id

{'32': 'CRC_2',
 '31': 'CRC_1',
 '35': 'CRC_4',
 '33': 'CRC_3',
 '38': 'CRC_7',
 '36': 'CRC_5',
 '37': 'CRC_6'}

In [20]:
ad.obs['BC_PatientID'] = ad.obs['PatientNumber'].map(patient_number_to_LC_id)
ad.obs

,Cell,nGene,nUMI,CellFromTumor,PatientNumber,TumorType,TumorSite,CellType,doublet_score,predicted_doublet,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,Project_ID,Primary_or_Metastatic,BC_PatientID
scrEXT001_AAACCTGGTCGGCTCA,scrEXT001_AAACCTGGTCGGCTCA,345,633,True,31,CRC,C,Cancer,0.031686,False,345,633.0,88.0,13.902053,2098-Colorectalcancer,Primary,CRC_1
scrEXT001_AAACCTGGTCTTTCAT,scrEXT001_AAACCTGGTCTTTCAT,2640,8119,True,31,CRC,C,Cancer,0.067423,False,2640,8119.0,1261.0,15.531470,2098-Colorectalcancer,Primary,CRC_1
scrEXT001_AAACGGGGTATATGGA,scrEXT001_AAACGGGGTATATGGA,364,602,True,31,CRC,C,Cancer,0.042946,False,364,602.0,2.0,0.332226,2098-Colorectalcancer,Primary,CRC_1
scrEXT001_AAACGGGTCGGTTAAC,scrEXT001_AAACGGGTCGGTTAAC,628,1058,True,31,CRC,C,Cancer,0.062164,False,628,1058.0,34.0,3.213610,2098-Colorectalcancer,Primary,CRC_1
scrEXT001_AAAGATGGTATAGGGC,scrEXT001_AAAGATGGTATAGGGC,3308,10935,True,31,CRC,C,Cancer,0.052468,False,3308,10935.0,739.0,6.758116,2098-Colorectalcancer,Primary,CRC_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
scrEXT029_TGCGGGTTCTCACATT,scrEXT029_TGCGGGTTCTCACATT,224,488,False,38,CRC,N,Cancer,0.030555,False,224,488.0,83.0,17.008198,2098-Colorectalcancer,Primary,CRC_7
scrEXT029_TGGGCGTGTCGGATCC,scrEXT029_TGGGCGTGTCGGATCC,265,444,False,38,CRC,N,Cancer,0.044566,False,265,444.0,72.0,16.216215,2098-Colorectalcancer,Primary,CRC_7
scrEXT029_TTCTCAACACAGGAGT,scrEXT029_TTCTCAACACAGGAGT,492,1202,False,38,CRC,N,Cancer,0.052468,False,492,1202.0,61.0,5.074875,2098-Colorectalcancer,Primary,CRC_7
scrEXT029_TTGGAACAGTGGACGT,scrEXT029_TTGGAACAGTGGACGT,460,931,False,38,CRC,N,Cancer,0.057429,False,460,931.0,123.0,13.211600,2098-Colorectalcancer,Primary,CRC_7


In [21]:
patient_meta_df = pd.read_csv('../../Data/BRCA/2102-Breastcancer/Patient_metadata_S1.txt', sep='\t')
patient_meta_df = patient_meta_df[patient_meta_df['Tumor_type'] == 'CRC']
meta_subset = patient_meta_df
meta_subset

,Patient_number,Tumor_type,Gender,Age_range,Stage,TNM,Pathological_subtype,Molecular_status
13,CRC_1,CRC,Female,80-85,IIB,pT4aN0M0,"Right caecum, moderately differentiated adenoc...",MSI-high
14,CRC_2,CRC,Female,86-90,IIIB,pT3N1bM0,"Left rectosigmoid, moderately differentiated a...",MSS
15,CRC_3,CRC,Female,50-55,IVA,pT4aN1aM1a,"Left sigmoid, moderately differentiated adenoc...",MSS
16,CRC_4,CRC,Male,80-85,I,pT2N0M0,"Left sigmoid, moderately differentiated adenoc...",MSS
17,CRC_5,CRC,Male,50-55,IIA,pT3N0L1,"Left sigmoid, moderately differentiated adenoc...",MSS
18,CRC_6,CRC,Female,76-80,IIIB,pT3N1aM0,"Right caecum, moderately differentiated adenoc...",MSS
19,CRC_7,CRC,Male,80-85,IIA,pT3N0L1,"Right ascending, moderately differentiated ade...",MSS


In [22]:
ad.obs = ad.obs.merge(meta_subset, left_on='BC_PatientID', right_on='Patient_number', how='left')
ad.obs

,Cell,nGene,nUMI,CellFromTumor,PatientNumber,TumorType,TumorSite,CellType,doublet_score,predicted_doublet,...,Primary_or_Metastatic,BC_PatientID,Patient_number,Tumor_type,Gender,Age_range,Stage,TNM,Pathological_subtype,Molecular_status
0,scrEXT001_AAACCTGGTCGGCTCA,345,633,True,31,CRC,C,Cancer,0.031686,False,...,Primary,CRC_1,CRC_1,CRC,Female,80-85,IIB,pT4aN0M0,"Right caecum, moderately differentiated adenoc...",MSI-high
1,scrEXT001_AAACCTGGTCTTTCAT,2640,8119,True,31,CRC,C,Cancer,0.067423,False,...,Primary,CRC_1,CRC_1,CRC,Female,80-85,IIB,pT4aN0M0,"Right caecum, moderately differentiated adenoc...",MSI-high
2,scrEXT001_AAACGGGGTATATGGA,364,602,True,31,CRC,C,Cancer,0.042946,False,...,Primary,CRC_1,CRC_1,CRC,Female,80-85,IIB,pT4aN0M0,"Right caecum, moderately differentiated adenoc...",MSI-high
3,scrEXT001_AAACGGGTCGGTTAAC,628,1058,True,31,CRC,C,Cancer,0.062164,False,...,Primary,CRC_1,CRC_1,CRC,Female,80-85,IIB,pT4aN0M0,"Right caecum, moderately differentiated adenoc...",MSI-high
4,scrEXT001_AAAGATGGTATAGGGC,3308,10935,True,31,CRC,C,Cancer,0.052468,False,...,Primary,CRC_1,CRC_1,CRC,Female,80-85,IIB,pT4aN0M0,"Right caecum, moderately differentiated adenoc...",MSI-high
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9034,scrEXT029_TGCGGGTTCTCACATT,224,488,False,38,CRC,N,Cancer,0.030555,False,...,Primary,CRC_7,CRC_7,CRC,Male,80-85,IIA,pT3N0L1,"Right ascending, moderately differentiated ade...",MSS
9035,scrEXT029_TGGGCGTGTCGGATCC,265,444,False,38,CRC,N,Cancer,0.044566,False,...,Primary,CRC_7,CRC_7,CRC,Male,80-85,IIA,pT3N0L1,"Right ascending, moderately differentiated ade...",MSS
9036,scrEXT029_TTCTCAACACAGGAGT,492,1202,False,38,CRC,N,Cancer,0.052468,False,...,Primary,CRC_7,CRC_7,CRC,Male,80-85,IIA,pT3N0L1,"Right ascending, moderately differentiated ade...",MSS
9037,scrEXT029_TTGGAACAGTGGACGT,460,931,False,38,CRC,N,Cancer,0.057429,False,...,Primary,CRC_7,CRC_7,CRC,Male,80-85,IIA,pT3N0L1,"Right ascending, moderately differentiated ade...",MSS


In [23]:
# Ensure TNM is string
ad.obs['TNM'] = ad.obs['TNM'].astype(str)

# If Primary_or_Metastatic is categorical, add new category first
if pd.api.types.is_categorical_dtype(ad.obs['Primary_or_Metastatic']):
    ad.obs['Primary_or_Metastatic'] = ad.obs['Primary_or_Metastatic'].cat.add_categories(['Metastatic'])

# Now assign "Metastatic" to rows where TNM contains "M1"
ad.obs.loc[ad.obs['TNM'].str.contains('M1', na=False), 'Primary_or_Metastatic'] = 'Metastatic'


In [24]:
ad.obs['Primary_or_Metastatic'].value_counts()

Primary_or_Metastatic
Primary       6826
Metastatic    2213
Name: count, dtype: int64

In [25]:
ad.obs['Final_cancer_type'] = 'Colorectal Cancer'
ad.obs['Final_histological_subtype'] = ad.obs.Pathological_subtype
ad.obs['Final_molecular_subtype'] = ad.obs.Molecular_status
ad.obs['Final_tissue'] = 'Colon'
ad.obs['Final_sample_id'] = ad.obs['BC_PatientID']

In [26]:
ad

AnnData object with n_obs × n_vars = 9039 × 33694
    obs: 'Cell', 'nGene', 'nUMI', 'CellFromTumor', 'PatientNumber', 'TumorType', 'TumorSite', 'CellType', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic', 'BC_PatientID', 'Patient_number', 'Tumor_type', 'Gender', 'Age_range', 'Stage', 'TNM', 'Pathological_subtype', 'Molecular_status', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue', 'Final_sample_id'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    uns: 'scrublet', 'log1p', 'pca', 'neighbors', 'umap', 'PatientNumber_colors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [27]:
# add more clinical information
ad.obs['Final_patient_age'] = ad.obs['Age_range']
ad.obs['Final_patient_stage'] = ad.obs['TNM']
ad.obs['Final_patient_treatment'] = 'Naïve'

In [28]:
ad.raw.shape

(9039, 33694)

In [29]:
ad

AnnData object with n_obs × n_vars = 9039 × 33694
    obs: 'Cell', 'nGene', 'nUMI', 'CellFromTumor', 'PatientNumber', 'TumorType', 'TumorSite', 'CellType', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic', 'BC_PatientID', 'Patient_number', 'Tumor_type', 'Gender', 'Age_range', 'Stage', 'TNM', 'Pathological_subtype', 'Molecular_status', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue', 'Final_sample_id', 'Final_patient_age', 'Final_patient_stage', 'Final_patient_treatment'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    uns: 'scrublet', 'log1p', 'pca', 'neighbors', 'umap', 'PatientNumber_colors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [30]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/2098-Colorectalcancer.COAD.h5ad', compression='gzip')

## Single-cell and spatial transcriptome analysis reveals the cellular heterogeneity of liver metastatic colorectal cancer


Paper: https://www.science.org/doi/full/10.1126/sciadv.adf5464?rfr_dat=cr_pub++0pubmed&url_ver=Z39.88-2003&rfr_id=ori%3Arid%3Acrossref.org#sec-4

Data downloaded from: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE225857

Data File: 
- GSM7058755_non_immune_counts.txt.gz
- GSM7058755_non_immune_meta.txt.gz


In [31]:
file_path = '../../Data/COAD/GSE225857/GSM7058755_non_immune_counts.txt'

In [32]:
# Step 1: Read just the first line to get cell names
with open(file_path) as f:
    header = f.readline().strip().split('\t')
cell_names = header[1:]  # skip the gene column
print(cell_names[:10])
# Step 2: Read file in chunks and store rows
gene_names = []
data = []

# count = 0
with open(file_path) as f:
    next(f)  # skip header
    for line in tqdm(f, desc="Reading rows"):
        parts = line.strip().split('\t')
        gene = parts[0]
        counts = np.array(parts[1:], dtype=np.float32)
        gene_names.append(gene)
        data.append(counts)
print(gene_names[:10])

# Step 3: Stack into matrix and transpose
dense_matrix = np.vstack(data)  # shape: (genes, cells)
transposed = sp.csr_matrix(dense_matrix.T)  # shape: (cells, genes)

# Step 4: Create AnnData
ad = ann.AnnData(X=transposed,
                   obs=pd.DataFrame(index=cell_names),
                   var=pd.DataFrame(index=gene_names))
ad

['"s1231_SampleTag06.710210s4"', '"s1231_SampleTag06.527732s4"', '"s1231_SampleTag06.384716s4"', '"s1231_SampleTag06.724127s4"', '"s1231_SampleTag06.309004s4"', '"s1231_SampleTag06.165204s4"', '"s1231_SampleTag06.129567s4"', '"s1231_SampleTag05.594586s4"', '"s1231_SampleTag05.762875s4"', '"s1231_SampleTag06.52287s4"']


Reading rows: 17515it [01:00, 290.76it/s]


['"A1BG"', '"A1CF"', '"A2M"', '"A2ML1"', '"A4GALT"', '"AAAS"', '"AACS"', '"AADAC"', '"AADAT"', '"AAGAB"']


AnnData object with n_obs × n_vars = 41892 × 17515

In [33]:
ad.obs_names = ad.obs_names.str.replace('"', '', regex=False).str.replace('.', '-')
ad.var_names = ad.var_names.str.replace('"', '', regex=False)

In [34]:
ad.to_df()

,A1BG,A1CF,A2M,A2ML1,A4GALT,AAAS,AACS,AADAC,AADAT,AAGAB,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
s1231_SampleTag06-710210s4,0.0,3.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0
s1231_SampleTag06-527732s4,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0,6.0,1.0
s1231_SampleTag06-384716s4,0.0,3.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,15.0,0.0
s1231_SampleTag06-724127s4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
s1231_SampleTag06-309004s4,0.0,0.0,9.0,0.0,3.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,14.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
s0107_SampleTag05-763220s19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
s0107_SampleTag05-637097s19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
s0107_SampleTag12-269718s19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
s0107_SampleTag05-577650s19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [35]:
metadata_df = pd.read_csv('../../Data/COAD/GSE225857/GSM7058755_non_immune_meta.txt', sep='\t', index_col=0)
metadata_df

,orig.ident,nCount_RNA,nFeature_RNA,patients,sampletag,organs,percent.mt,percent.ribo,log10GenesPerUMI,integrated_snn_res.0.5,seurat_clusters,batch,doublet.score,predicted.doublet,doublet,cluster,integrated_snn_res.0.1,patients_organ
s1231_SampleTag06-710210s4,s1231,24851,4688,s1231,s1231_SampleTag06,LCT,32.646574,11.677598,0.835199,2,0,s1231,0.042424,False,singlet,Tu01_AREG,0,LCT_s1231
s1231_SampleTag06-527732s4,s1231,24851,4832,s1231,s1231_SampleTag06,LCT,23.085590,13.987365,0.838189,6,0,s1231,0.104532,False,singlet,Tu10_COL3A1,0,LCT_s1231
s1231_SampleTag06-384716s4,s1231,24067,4443,s1231,s1231_SampleTag06,LCT,24.290522,7.229817,0.832533,6,0,s1231,0.044103,False,singlet,Tu01_AREG,0,LCT_s1231
s1231_SampleTag06-724127s4,s1231,24689,4739,s1231,s1231_SampleTag06,LCT,31.560614,6.517072,0.836809,16,7,s1231,0.011252,False,singlet,Tu08_GNG13,7,LCT_s1231
s1231_SampleTag06-309004s4,s1231,24676,5223,s1231,s1231_SampleTag06,LCT,7.748420,7.286432,0.846468,7,2,s1231,0.117452,False,singlet,F02_fibrblast_MCAM,2,LCT_s1231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
s0107_SampleTag05-763220s19,s0107,859,299,s0107,s0107_SampleTag05,CCT,26.775320,9.080326,0.843789,19,0,s0107,0.010608,False,singlet,Tu01_AREG,0,CCT_s0107
s0107_SampleTag05-637097s19,s0107,956,414,s0107,s0107_SampleTag05,CCT,38.075314,1.255230,0.878053,3,4,s0107,0.123077,False,singlet,Tu03_SRRM2,4,CCT_s0107
s0107_SampleTag12-269718s19,s0107,924,489,s0107,s0107_SampleTag12,LCT,1.948052,22.510823,0.906813,3,4,s0107,0.025641,False,singlet,Tu04_RGMB,4,LCT_s0107
s0107_SampleTag05-577650s19,s0107,820,399,s0107,s0107_SampleTag05,CCT,36.219512,10.975610,0.892635,3,4,s0107,0.050949,False,singlet,Tu03_SRRM2,4,CCT_s0107


In [36]:
ad.obs = metadata_df.loc[ad.obs.index]
ad

AnnData object with n_obs × n_vars = 41892 × 17515
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'patients', 'sampletag', 'organs', 'percent.mt', 'percent.ribo', 'log10GenesPerUMI', 'integrated_snn_res.0.5', 'seurat_clusters', 'batch', 'doublet.score', 'predicted.doublet', 'doublet', 'cluster', 'integrated_snn_res.0.1', 'patients_organ'

In [37]:
ad.raw = ad

In [38]:
# re-process the adata
ad = reprocess_from_raw_layer(ad, 
                              Project_ID='GSE225857', 
                              Primary_or_Metastatic='Metastatic',
                              further_pre=True)

Running doublet detection on 41892 cells...


/home/wang3712/.local/lib/python3.8/site-packages/scanpy/preprocessing/_normalization.py:170: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


Automatically set threshold at doublet score = 0.54
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 19.5%
Overall doublet rate:
	Expected   = 6.0%
	Estimated  = 0.5%
  Detected 38 doublets (0.1%)
  After doublet removal: 41854 cells
Standard filtering...
Normalizing...
Scaling...
Computing PCA...
Computing neighbors...
Computing UMAP...
Reprocessed dataset. Final shape: (41828, 17515)


In [ ]:
for obs in ['patients', 'organs', 'cluster']:
    sc.pl.umap(ad, color=obs)

In [40]:
tumor_types = []
for i in ad.obs.cluster.value_counts().index:
    if i.startswith('Tu'):
        tumor_types.append(i)
tumor_types

['Tu02_DEFA5',
 'Tu01_AREG',
 'Tu03_SRRM2',
 'Tu06_NKD1',
 'Tu05_PCNA',
 'Tu04_RGMB',
 'Tu07_MKI67',
 'Tu08_GNG13',
 'Tu10_COL3A1',
 'Tu09_MUC2',
 'Tu11_PLA2G2A']

In [41]:
ad = filter_and_recompute(adata=ad, 
                          celltype_col='cluster', 
                          celltypes_to_keep=tumor_types,
                          further_pre=True)

Original shape: (41828, 17515)
Filtered shape: (23908, 17515)
Recalculated PCA and UMAP.


In [ ]:
for obs in ['patients', 'organs', 'cluster']:
    sc.pl.umap(ad, color=obs)

In [43]:
ad

AnnData object with n_obs × n_vars = 23908 × 17515
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'patients', 'sampletag', 'organs', 'percent.mt', 'percent.ribo', 'log10GenesPerUMI', 'integrated_snn_res.0.5', 'seurat_clusters', 'batch', 'doublet.score', 'predicted.doublet', 'doublet', 'cluster', 'integrated_snn_res.0.1', 'patients_organ', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    uns: 'scrublet', 'log1p', 'pca', 'neighbors', 'umap', 'patients_colors', 'organs_colors', 'cluster_colors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [44]:
new_tissues = []
for tissue in ad.obs.organs:
    if tissue == 'CCT':
        new_tissues.append('Colon')
    elif tissue == 'LCT':
        new_tissues.append('Liver')
    else:
        print('Wrong')

In [45]:
ad.obs['Final_cancer_type'] = 'Colorectal Cancer'
ad.obs['Final_histological_subtype'] = 'COAD: Unspecified'
ad.obs['Final_molecular_subtype'] = 'COAD: Unspecified'
ad.obs['Final_tissue'] = new_tissues
ad.obs['Final_sample_id'] = ad.obs['patients']

In [46]:
# Clinical + Treatment metadata (merged)
patient_metadata = {
    "s0107": {"Age": 75, "Gender": "Male", "Primary site": "Left colon", "Tissues": ["CN", "CC", "LN", "LM", "PB"], "Regimen": "FOLFOX", "Cycles": 3},
    "s0115": {"Age": 80, "Gender": "Male", "Primary site": "Left colon", "Tissues": ["CN", "CC", "LN", "LM", "PB"], "Regimen": "FOLFOX", "Cycles": 4},
    "s0813": {"Age": 58, "Gender": "Female", "Primary site": "Right colon", "Tissues": ["CN", "CC", "LN", "LM", "PB"], "Regimen": "FOLFOXIRI", "Cycles": 5},
    "s0920": {"Age": 64, "Gender": "Male", "Primary site": "Left colon", "Tissues": ["CC", "LN", "LM", "PB"], "Regimen": "FOLFOX", "Cycles": 5},
    "s1125": {"Age": 41, "Gender": "Female", "Primary site": "Left colon", "Tissues": ["CN", "CC", "LN", "LM"], "Regimen": "FOLFOXIRI+Bevacizumab", "Cycles": 8},
    "s1231": {"Age": 58, "Gender": "Male", "Primary site": "Left colon", "Tissues": ["CN", "CC", "LN", "LM", "PB"], "Regimen": "FOLFOX", "Cycles": 3}
}

In [47]:
# Extract dictionaries from the metadata you want to add
age_map = {k: v["Age"] for k, v in patient_metadata.items()}
# stage_map = {k: v.get("Cycles", None) for k, v in patient_metadata.items()}  # or use another field if you meant clinical stage
treatment_map = {k: v["Regimen"] for k, v in patient_metadata.items()}


In [48]:
# add more clinical information
ad.obs['Final_patient_age'] = ad.obs['patients'].map(age_map)
ad.obs['Final_patient_stage'] = 'Unknown'
ad.obs['Final_patient_treatment'] = ad.obs['patients'].map(treatment_map)

In [ ]:
for obs in ['Final_patient_age', 'Final_patient_treatment']:
    sc.pl.umap(ad, color=obs)

In [50]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/GSE225857.COAD.h5ad', compression='gzip')

## Spatially organized multicellular immune hubs in human colorectal cancer


Paper: https://www.cell.com/cell/fulltext/S0092-8674(21)00945-4

Data downloaded from: 
- https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE178341
- https://singlecell.broadinstitute.org/single_cell/study/SCP1162/human-colon-cancer-atlas-c295#study-download

Files: 
- Matrix:https://ftp.ncbi.nlm.nih.gov/geo/series/GSE178nnn/GSE178341/suppl/GSE178341%5Fcrc10x%5Ffull%5Fc295v4%5Fsubmit.h5 
- Annotation: 
-- https://singlecell.broadinstitute.org/single_cell/data/public/SCP1162/human-colon-cancer-atlas-c295?filename=metatable_v3_fix_v3.tsv
-- https://singlecell.broadinstitute.org/single_cell/data/public/SCP1162/human-colon-cancer-atlas-c295?filename=crc10x_tSNE_cl_global.tsv


In [10]:
# Step 1: Load from the .h5 file
with h5py.File('../../Data/COAD/GSE178341/GSE178341_crc10x_full_c295v4_submit.h5', 'r') as f:
    mat = f['matrix']
    
    # Sparse matrix components
    data = mat['data'][:]
    indices = mat['indices'][:]
    indptr = mat['indptr'][:]
    shape = tuple(mat['shape'])  # e.g. (genes, cells)

    # Cell barcodes and feature (gene) names
    barcodes = mat['barcodes'][:].astype(str)
    features = mat['features']['name'][:].astype(str)

# Step 2: Build sparse matrix
X = sp.csc_matrix((data, indices, indptr), shape=shape)

# Step 3: Create AnnData object
# By convention: cells = obs = rows, genes = vars = columns → transpose
ad = ann.AnnData(X.T)
ad.var_names = features
ad.obs_names = barcodes
ad

AnnData object with n_obs × n_vars = 370115 × 43113

In [11]:
memory_usgae()
del X, data
memory_usgae()

Current memory usage: 9.31 GB
Current memory usage: 9.31 GB


In [12]:
ad.raw = ad

In [13]:
metadata_df = pd.read_csv('../../Data/COAD/GSE178341/metatable_v3_fix_v3.tsv', sep='\t', index_col=0)
metadata_df

/tmp/ipykernel_1358960/757064394.py:1: DtypeWarning: Columns (14,15,16,17,18,19,20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata_df = pd.read_csv('../../Data/COAD/GSE178341/metatable_v3_fix_v3.tsv', sep='\t', index_col=0)


,biosample_id,donor_id,SpecimenType,TissueSource,ProcessingMethod,PatientTypeID,sex,Site,Grade,TumorStage,...,qc_emptyDropPval,qc_mitoFraction,species,species__ontology_label,disease,disease__ontology_label,organ,organ__ontology_label,library_preparation_protocol,library_preparation_protocol__ontology_label
NAME,,,,,,,,,,,,,,,,,,,,,
TYPE,group,group,group,group,group,group,group,group,group,group,...,numeric,numeric,group,group,group,group,group,group,group,group
C103_T_1_1_0_c1_v2_id-AAACCTGCATGCTAGT,C103_T_1_1_0_c1_v2,C103,T,MGH,unsorted,C103_T,male,left,low,notT4,...,9.99900009999e-05,0.0762605181209832,NCBITaxon_9606,Homo sapiens,MONDO_0002271,colon adenocarcinoma,UBERON_0001155,colon,EFO_0009899,10X 3' v2 sequencing
C103_T_1_1_0_c1_v2_id-AAACCTGGTAGCCTAT,C103_T_1_1_0_c1_v2,C103,T,MGH,unsorted,C103_T,male,left,low,notT4,...,9.99900009999e-05,0.352076124567474,NCBITaxon_9606,Homo sapiens,MONDO_0002271,colon adenocarcinoma,UBERON_0001155,colon,EFO_0009899,10X 3' v2 sequencing
C103_T_1_1_0_c1_v2_id-AAACCTGGTTGTCGCG,C103_T_1_1_0_c1_v2,C103,T,MGH,unsorted,C103_T,male,left,low,notT4,...,9.99900009999e-05,0.112033277605286,NCBITaxon_9606,Homo sapiens,MONDO_0002271,colon adenocarcinoma,UBERON_0001155,colon,EFO_0009899,10X 3' v2 sequencing
C103_T_1_1_0_c1_v2_id-AAACCTGTCATGTGGT,C103_T_1_1_0_c1_v2,C103,T,MGH,unsorted,C103_T,male,left,low,notT4,...,9.99900009999e-05,0.108513779527559,NCBITaxon_9606,Homo sapiens,MONDO_0002271,colon adenocarcinoma,UBERON_0001155,colon,EFO_0009899,10X 3' v2 sequencing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
C173_T_0_0_0_c1_v3_id-TTTGGAGTCATCGGGC,C173_T_0_0_0_c1_v3,C173,T,DFCI,unsorted,C173_T,female,left,high,T4,...,0.0001,0.108428,NCBITaxon_9606,Homo sapiens,MONDO_0002271,colon adenocarcinoma,UBERON_0001155,colon,EFO_0009922,10x 3' v3 sequencing
C173_T_0_0_0_c1_v3_id-TTTGGAGTCTAGTGTG,C173_T_0_0_0_c1_v3,C173,T,DFCI,unsorted,C173_T,female,left,high,T4,...,0.0001,0.260756,NCBITaxon_9606,Homo sapiens,MONDO_0002271,colon adenocarcinoma,UBERON_0001155,colon,EFO_0009922,10x 3' v3 sequencing
C173_T_0_0_0_c1_v3_id-TTTGTTGCAGCAATTC,C173_T_0_0_0_c1_v3,C173,T,DFCI,unsorted,C173_T,female,left,high,T4,...,0.0001,0.437101,NCBITaxon_9606,Homo sapiens,MONDO_0002271,colon adenocarcinoma,UBERON_0001155,colon,EFO_0009922,10x 3' v3 sequencing


In [14]:
ad.obs = metadata_df.loc[ad.obs.index]
ad

AnnData object with n_obs × n_vars = 370115 × 43113
    obs: 'biosample_id', 'donor_id', 'SpecimenType', 'TissueSource', 'ProcessingMethod', 'PatientTypeID', 'sex', 'Site', 'Grade', 'TumorStage', 'LymphNodeStatus', 'MMRStatusTumor', 'MMRMLH1Tumor', 'qc_geneCount', 'qc_logMappedReads', 'qc_meanReadsPerUmi', 'qc_totalReads', 'qc_logUmiCount', 'qc_bcSwapFraction', 'qc_geneSatFraction', 'qc_seqDupEst', 'qc_umiSatFraction', 'qc_emptyDropPval', 'qc_mitoFraction', 'species', 'species__ontology_label', 'disease', 'disease__ontology_label', 'organ', 'organ__ontology_label', 'library_preparation_protocol', 'library_preparation_protocol__ontology_label'

In [15]:
metadata_df = pd.read_csv('../../Data/COAD/GSE178341/crc10x_tSNE_cl_global.txt', sep='\t', index_col=0)

In [16]:
ad.obs['ClusterMidway'] = metadata_df.loc[ad.obs.index]['ClusterMidway']

In [17]:
ad = ad[ad.obs['ClusterMidway'].isin(set(['EpiT']))].copy()

/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [18]:
ad

AnnData object with n_obs × n_vars = 108131 × 43113
    obs: 'biosample_id', 'donor_id', 'SpecimenType', 'TissueSource', 'ProcessingMethod', 'PatientTypeID', 'sex', 'Site', 'Grade', 'TumorStage', 'LymphNodeStatus', 'MMRStatusTumor', 'MMRMLH1Tumor', 'qc_geneCount', 'qc_logMappedReads', 'qc_meanReadsPerUmi', 'qc_totalReads', 'qc_logUmiCount', 'qc_bcSwapFraction', 'qc_geneSatFraction', 'qc_seqDupEst', 'qc_umiSatFraction', 'qc_emptyDropPval', 'qc_mitoFraction', 'species', 'species__ontology_label', 'disease', 'disease__ontology_label', 'organ', 'organ__ontology_label', 'library_preparation_protocol', 'library_preparation_protocol__ontology_label', 'ClusterMidway'

In [19]:
# re-process the adata
ad = reprocess_from_raw_layer(ad, 
                              Project_ID='GSE178341', 
                              Primary_or_Metastatic='Metastatic',
                              further_pre=True)

Running doublet detection on 108131 cells...


/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique

Automatically set threshold at doublet score = 0.78
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 0.1%
Overall doublet rate:
	Expected   = 6.0%
	Estimated  = 0.8%
  Detected 1 doublets (0.0%)


/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  After doublet removal: 108130 cells
Standard filtering...


/home/wang3712/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Normalizing...
Scaling...
Computing PCA...
Computing neighbors...


/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1086: NumbaDeprecationWarning:

Computing UMAP...
Reprocessed dataset. Final shape: (46201, 43113)


In [ ]:
reset_plot()
for obs in ['LymphNodeStatus', 'disease', 'species__ontology_label']:
    sc.pl.umap(ad, color=obs)

In [21]:
ad

AnnData object with n_obs × n_vars = 46201 × 43113
    obs: 'biosample_id', 'donor_id', 'SpecimenType', 'TissueSource', 'ProcessingMethod', 'PatientTypeID', 'sex', 'Site', 'Grade', 'TumorStage', 'LymphNodeStatus', 'MMRStatusTumor', 'MMRMLH1Tumor', 'qc_geneCount', 'qc_logMappedReads', 'qc_meanReadsPerUmi', 'qc_totalReads', 'qc_logUmiCount', 'qc_bcSwapFraction', 'qc_geneSatFraction', 'qc_seqDupEst', 'qc_umiSatFraction', 'qc_emptyDropPval', 'qc_mitoFraction', 'species', 'species__ontology_label', 'disease', 'disease__ontology_label', 'organ', 'organ__ontology_label', 'library_preparation_protocol', 'library_preparation_protocol__ontology_label', 'ClusterMidway', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    uns: 'scrublet', 'log1p', 'pca', 'neighbors', 'umap', 'LymphNodeStatus_co

In [22]:
patient_clinical_df = pd.read_csv('../../Data/COAD/GSE178341/Petient_clinical.txt', sep='\t')
patient_clinical_df

,PatientBarcode,SpecimenType,PatientBarcode_SpecimenType,Sex,Age,MMR-IHC,MMRStatus,MLH1Status,MMRMLH1Tumor,HistologicTypeSimple,...,Tumor Stage Raw (on resection specimen path report),Tumor Stage,Node Status Raw (on resection specimen path report),Node Status,Metastasis stage (on resection specimen path report),Size (Tumor Largest Dimension (cm)),Size Quantile,DaysDiagnosisToLastAccess,SurvivalStatus,Time to Death (days)
0,C103,T,C103_T,M,45,MSS,MMRp,MLH1NoMeth,MSS,Adenocarcinoma,...,pT2,2,N0,Nneg,not entered (Mx),2.5,1,1043,Alive,NaN
1,C104,T,C104_T,M,81,MSS,MMRp,MLH1NoMeth,MSS,Adenocarcinoma,...,pT3,3,N0,Nneg,not entered (Mx),4.0,2,1055,Alive,NaN
2,C105,T,C105_T,M,71,MSS,MMRp,MLH1NoMeth,MSS,Adenocarcinoma,...,pT4a,4,N1a,Npos,not entered (Mx),4.7,2,1033,Alive,NaN
3,C106,T,C106_T,M,67,MLH1 and PMS2 deficient,MMRd,MLH1Meth,MSI_MLH1Meth,Adenocarcinoma;Mucinous,...,pT4a,4,N0,Nneg,not entered (Mx),4.6,2,1018,Alive,NaN
4,C107,T,C107_T,M,62,MSS,MMRp,MLH1NoMeth,MSS,Adenocarcinoma,...,pT3,3,N1b,Npos,not entered (Mx),8.2,4,1040,Alive,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,C170,T,C170_T,F,77,MLH1 and PMS2 deficient,MMRd,MLH1Meth,MSI_MLH1Meth,Adenocarcinoma,...,pT3,3,N0,Nneg,not entered (Mx),6.3,3,658,Alive,NaN
60,C171,TA,C171_TA,M,61,MSS,MMRp,MLH1NoMeth,MSS,Adenocarcinoma,...,pT2,2,N0,Nneg,not entered (Mx),3.2,1,123,Alive,NaN
61,C171,TB,C171_TB,M,61,MSS,MMRp,MLH1NoMeth,MSS,Adenocarcinoma,...,pT2,2,N0,Nneg,not entered (Mx),3.2,1,123,Alive,NaN
62,C172,T,C172_T,F,61,MSS,MMRp,MLH1NoMeth,MSS,Adenocarcinoma,...,pT3,3,pN0,Nneg,not entered (Mx),5.0,3,619,Alive,NaN


In [24]:
# Step 1: Reset index for merging, but save cell IDs
merged = ad.obs.reset_index().merge(
    patient_clinical_df,
    left_on='PatientTypeID',
    right_on='PatientBarcode_SpecimenType',
    how='left'
)

# Step 2: Restore original index (cell barcodes)
merged = merged.set_index('index')

# Step 3: Assign back
ad.obs = merged

In [27]:
ad.obs['Project_ID'] = 'GSE178341'
ad.obs['Primary_or_Metastatic'] = ad.obs["Metastasis stage (on resection specimen path report)"].astype(str).apply(
    lambda x: "Metastatic" if "M1" in x else "M0"
)

In [28]:
ad.obs['Final_cancer_type'] = 'Colorectal Cancer'
ad.obs['Final_histological_subtype'] = ['COAD: '+i for i in ad.obs['HistologicTypeSimple']]
ad.obs['Final_molecular_subtype'] = ['COAD: '+i for i in ad.obs['MMRStatus']]
ad.obs['Final_tissue'] = 'Colon'
ad.obs['Final_sample_id'] = ad.obs['donor_id']

In [29]:
ad.raw.shape

(46201, 43113)

In [30]:
all_qc_obs = []
for col in ad.obs.columns:
    if col.startswith('qc_'):
        all_qc_obs.append(col)
all_qc_obs

['qc_geneCount',
 'qc_logMappedReads',
 'qc_meanReadsPerUmi',
 'qc_totalReads',
 'qc_logUmiCount',
 'qc_bcSwapFraction',
 'qc_geneSatFraction',
 'qc_seqDupEst',
 'qc_umiSatFraction',
 'qc_emptyDropPval',
 'qc_mitoFraction']

In [31]:
ad.obs = ad.obs.drop(columns=all_qc_obs)


In [35]:
# add more clinical information
ad.obs['Final_patient_age'] = ad.obs.Age
ad.obs['Final_patient_stage'] = ad.obs['Tumor Stage Raw (on resection specimen path report)']
ad.obs['Final_patient_treatment'] = 'Naïve'

In [36]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/GSE178341.COAD.h5ad', compression='gzip')

## Lineage-dependent gene expression programs influence the immune landscape of colorectal cancer.

Paper: https://www.nature.com/articles/s41588-020-0636-z#Fig2

Data downloaded from: 
- https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE132465

Files: 
- Matrix:https://ftp.ncbi.nlm.nih.gov/geo/series/GSE132nnn/GSE132465/suppl/GSE132465%5FGEO%5Fprocessed%5FCRC%5F10X%5Fraw%5FUMI%5Fcount%5Fmatrix.txt.gz
- Annotation: https://ftp.ncbi.nlm.nih.gov/geo/series/GSE132nnn/GSE132465/suppl/GSE132465%5FGEO%5Fprocessed%5FCRC%5F10X%5Fcell%5Fannotation.txt.gz
- Additional annotation: https://www.dropbox.com/scl/fi/go3m1x3j3gtmhmjexbvk6/Meta-data_Lee2020_Colorectal.tar.gz?rlkey=4p2qlwuz3ay3oztnun3b9kgfu&dl=1


In [10]:
file_path = '../../Data/COAD/GSE132465/GSE132465_GEO_processed_CRC_10X_raw_UMI_count_matrix.txt'

In [11]:
# Step 1: Read just the first line to get cell names
with open(file_path) as f:
    header = f.readline().strip().split('\t')
cell_names = header[1:]  # skip the gene column
print(cell_names[:10])
# Step 2: Read file in chunks and store rows
gene_names = []
data = []

# count = 0
with open(file_path) as f:
    next(f)  # skip header
    for line in tqdm(f, desc="Reading rows"):
        parts = line.strip().split('\t')
        gene = parts[0]
        counts = np.array(parts[1:], dtype=np.float32)
        gene_names.append(gene)
        data.append(counts)
print(gene_names[:10])


['SMC01-T_AAACCTGCATACGCCG', 'SMC01-T_AAACCTGGTCGCATAT', 'SMC01-T_AAACCTGTCCCTTGCA', 'SMC01-T_AAACGGGAGGGAAACA', 'SMC01-T_AAACGGGGTATAGGTA', 'SMC01-T_AAAGATGAGGCCGAAT', 'SMC01-T_AAAGATGCATGGATGG', 'SMC01-T_AAAGATGTCACGACTA', 'SMC01-T_AAAGATGTCCGTTGCT', 'SMC01-T_AAAGCAACAGTCGATT']


Reading rows: 33694it [02:52, 194.79it/s]

['A1BG', 'A1BG-AS1', 'A1CF', 'A2M', 'A2M-AS1', 'A2ML1', 'A2ML1-AS1', 'A2ML1-AS2', 'A3GALT2', 'A4GALT']


In [12]:

# Step 3: Stack into matrix and transpose
dense_matrix = np.vstack(data)  # shape: (genes, cells)
transposed = sp.csr_matrix(dense_matrix.T)  # shape: (cells, genes)

# Step 4: Create AnnData
ad = ann.AnnData(X=transposed,
                   obs=pd.DataFrame(index=cell_names),
                   var=pd.DataFrame(index=gene_names))
ad

AnnData object with n_obs × n_vars = 63689 × 33694

In [13]:
meta_df = pd.read_csv('../../Data/COAD/GSE132465/GSE132465_GEO_processed_CRC_10X_cell_annotation.txt', sep='\t', index_col=0)
meta_df

,Patient,Class,Sample,Cell_type,Cell_subtype
Index,,,,,
SMC01-T_AAACCTGCATACGCCG,SMC01,Tumor,SMC01-T,Epithelial cells,CMS2
SMC01-T_AAACCTGGTCGCATAT,SMC01,Tumor,SMC01-T,Epithelial cells,CMS2
SMC01-T_AAACCTGTCCCTTGCA,SMC01,Tumor,SMC01-T,Epithelial cells,CMS2
SMC01-T_AAACGGGAGGGAAACA,SMC01,Tumor,SMC01-T,Epithelial cells,CMS2
SMC01-T_AAACGGGGTATAGGTA,SMC01,Tumor,SMC01-T,Epithelial cells,CMS2
...,...,...,...,...,...
SMC10-N_TCAGCTCGTAGCGTCC,SMC10,Normal,SMC10-N,Mast cells,Mast cells
SMC10-N_TGACTAGCAGACGCAA,SMC10,Normal,SMC10-N,Mast cells,Mast cells
SMC10-N_TGCTACCGTCTCCATC,SMC10,Normal,SMC10-N,Mast cells,Mast cells


In [14]:
ad.obs = meta_df.loc[ad.to_df().index]

In [15]:
ad = ad[ad.obs['Class'] == 'Tumor']
ad

View of AnnData object with n_obs × n_vars = 47285 × 33694
    obs: 'Patient', 'Class', 'Sample', 'Cell_type', 'Cell_subtype'

In [16]:
ad.raw = ad

In [19]:
additional_meta_df = pd.read_csv('../../Data/COAD/GSE132465/Data_Lee2020_Colorectal/Cells.csv', index_col=0)
additional_meta_df

,sample,cell_type,complexity,umap1,umap2,g1s_score,g2m_score,cell_cycle_phase,mp_top_score,mp_top,mp_assignment
cell_name,,,,,,,,,,,
SMC01-T_AAACCTGAGAAGGTTT,SMC01,T_cell,1400,-27.1368,3.7871,-0.0747,0.1349,Not cycling,1.7135,CD4 - Treg,CD4 - Treg
SMC01-T_AAACCTGAGGTAGCTG,SMC01,T_cell,1249,-19.6810,7.0101,-0.0916,0.0023,Not cycling,0.7044,CD8 - Memory/Naive1,NaN
SMC01-T_AAACCTGCATACGCCG,SMC01,Malignant,4787,32.9617,2.7729,1.8071,1.1582,Intermediate,1.5992,Cell Cycle - G1/S,Cell Cycle - G1/S
SMC01-T_AAACCTGGTCGCATAT,SMC01,Malignant,5175,32.2640,1.7495,0.3591,0.0705,Not cycling,0.4732,Secreted I,NaN
SMC01-T_AAACCTGGTTCCTCCA,SMC01,T_cell,1245,-19.4004,13.4971,-0.1236,0.0481,Not cycling,1.3922,CD8 - Heat shock,CD8 - Heat shock
...,...,...,...,...,...,...,...,...,...,...,...
SMC25-T_TTTGCGCAGACACGAC,SMC25,Malignant,5016,-5.6256,13.0493,0.1811,-0.0489,Not cycling,0.4086,Secreted II,NaN
SMC25-T_TTTGCGCCATGGAATA,SMC25,NaN,3376,-6.0811,12.9868,NaN,NaN,NaN,NaN,NaN,NaN
SMC25-T_TTTGGTTCAACGCACC,SMC25,B_cell,1745,-5.6495,23.8983,-0.1133,-0.0472,Not cycling,NaN,NaN,NaN


In [20]:
shared_cells = list(set(additional_meta_df.index).intersection(set(ad.to_df().index)))
shared_cells.sort()
ad = ad[shared_cells]
ad

View of AnnData object with n_obs × n_vars = 21657 × 33694
    obs: 'Patient', 'Class', 'Sample', 'Cell_type', 'Cell_subtype'

In [21]:
ad.obs = additional_meta_df.loc[shared_cells]

In [22]:
# re-process the adata
ad = reprocess_from_raw_layer(ad, 
                              Project_ID='GSE132465', 
                              Primary_or_Metastatic='Metastatic',
                              further_pre=False)

Running doublet detection on 21657 cells...


/home/wang3712/.local/lib/python3.8/site-packages/scanpy/preprocessing/_normalization.py:170: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


Automatically set threshold at doublet score = 0.56
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 24.9%
Overall doublet rate:
	Expected   = 6.0%
	Estimated  = 0.5%
  Detected 28 doublets (0.1%)
  After doublet removal: 21629 cells
Standard filtering...
Reprocessed dataset. Final shape: (19830, 33694)


In [23]:
ad = filter_and_recompute(adata=ad, 
                          celltype_col='cell_type', 
                          celltypes_to_keep=['Malignant'],
                          further_pre=True)
ad

Original shape: (19830, 33694)
Filtered shape: (9350, 33694)


/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1086: NumbaDeprecationWarning:

Recalculated PCA and UMAP.


AnnData object with n_obs × n_vars = 9350 × 33694
    obs: 'sample', 'cell_type', 'complexity', 'umap1', 'umap2', 'g1s_score', 'g2m_score', 'cell_cycle_phase', 'mp_top_score', 'mp_top', 'mp_assignment', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'scrublet', 'pca', 'neighbors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [ ]:
reset_plot()
sc.pl.umap(ad, color=['sample'])

In [25]:
patient_clinical_df = pd.read_csv('../../Data/COAD/GSE132465/patient_clinical.txt', sep='\t')
patient_clinical_df

,Patient,Tumor,Sample type,Gender,Age,TNM stage,Stage,Anatomic region,Left/Right-sided,MSI,...,No.of mutations,nearestCMS (RF),predictedCMS (RF),KRAS,BRAF,TP53,APC,SMAD4,Alias (Tumor),Alias (Normal)
0,SMC01,SMC01-T,Colorectal cancer,F,64,T3 N0 M0,IIA,rectum,left,MSS,...,157,CMS3,NaN,Mutant,Wildtype,Mutant,Mutant,Wildtype,PM-PS-0001-T-A1,PM-PS-0001-N-A1
1,SMC02,SMC02-T,Colorectal cancer,M,66,T3 N1b M0,IIIB,rectum,left,MSS,...,105,CMS4,CMS4,Wildtype,Wildtype,Mutant,Mutant,Mutant,PM-PS-0002-T-A1,PM-PS-0002-N-A1
2,SMC03,SMC03-T,Colorectal cancer,F,83,T4b N2a M0,IIIC,hepatic flexure,right,MSI-H,...,6134,CMS1,CMS1,Wildtype,Mutant,Wildtype,Mutant,Wildtype,PM-PS-0003-T-A1,PM-PS-0003-N-A1
3,SMC04,SMC04-T,Colorectal cancer,M,69,T3 N1b M0,IIIB,sigmoid,left,MSS,...,86,CMS4,CMS4,Mutant,Wildtype,Mutant,Mutant,Wildtype,PM-PS-0004-T-A1,PM-PS-0004-N-A1
4,SMC05,SMC05-T,Colorectal cancer,F,58,T3 N0 M0,IIA,ascending,right,MSS,...,122,CMS3,CMS3,Mutant,Wildtype,Wildtype,Mutant,Wildtype,PM-PS-0005-T-A1,PM-PS-0005-N-A1
5,SMC06,SMC06-T,Colorectal cancer,M,46,T3 N1b M0,IIIB,hepatic flexure,right,MSI-H,...,1134,CMS1,CMS1,Mutant,Wildtype,Wildtype,Wildtype,Wildtype,PM-PS-0006-T-A1,PM-PS-0006-N-A1
6,SMC07,SMC07-T,Colorectal cancer,F,67,T2 N0 M0,I,ascending,right,MSS,...,103,CMS2,NaN,Mutant,Wildtype,Mutant,Mutant,Wildtype,PM-PS-0007-T-A1,PM-PS-0007-N-A1
7,SMC08,SMC08-T,Colorectal cancer,M,68,T3 N1b M0,IIIB,sigmoid,left,MSS,...,95,CMS1,NaN,Wildtype,Wildtype,Mutant,Mutant,Wildtype,PM-PS-0008-T-A1,PM-PS-0008-N-A1
8,SMC09,SMC09-T,Colorectal cancer,M,75,T3 N0 M0,IIA,sigmoid,left,MSS,...,112,CMS2,CMS2,Wildtype,Wildtype,Mutant,Mutant,Wildtype,PM-PS-0009-T-A1,PM-PS-0009-N-A1
9,SMC10,SMC10-T,Colorectal cancer,F,77,T3 N0 M0,IIA,ascending,right,MSI-H,...,987,CMS1,NaN,Wildtype,Mutant,Wildtype,Wildtype,Wildtype,PM-PS-0010-T-A1,PM-PS-0010-N-A1


In [26]:
merged = ad.obs.reset_index().merge(
    patient_clinical_df,
    left_on='sample',
    right_on='Patient',
    how='left'
)

# Step 2: Restore original index (cell barcodes)
merged = merged.set_index('cell_name')

# Step 3: Assign back
ad.obs = merged
ad.obs

,sample,cell_type,complexity,umap1,umap2,g1s_score,g2m_score,cell_cycle_phase,mp_top_score,mp_top,...,No.of mutations,nearestCMS (RF),predictedCMS (RF),KRAS,BRAF,TP53,APC,SMAD4,Alias (Tumor),Alias (Normal)
cell_name,,,,,,,,,,,,,,,,,,,,,
SMC01-T_AAACCTGCATACGCCG,SMC01,Malignant,4787,32.9617,2.7729,1.8071,1.1582,Intermediate,1.5992,Cell Cycle - G1/S,...,157,CMS3,NaN,Mutant,Wildtype,Mutant,Mutant,Wildtype,PM-PS-0001-T-A1,PM-PS-0001-N-A1
SMC01-T_AAACCTGTCCCTTGCA,SMC01,Malignant,1685,4.3380,0.1163,0.1708,0.1746,Not cycling,1.3451,MP41 (Unassigned),...,157,CMS3,NaN,Mutant,Wildtype,Mutant,Mutant,Wildtype,PM-PS-0001-T-A1,PM-PS-0001-N-A1
SMC01-T_AAACGGGGTATAGGTA,SMC01,Malignant,3849,30.5661,-0.6466,-0.1692,-0.0068,Not cycling,1.0476,Stress,...,157,CMS3,NaN,Mutant,Wildtype,Mutant,Mutant,Wildtype,PM-PS-0001-T-A1,PM-PS-0001-N-A1
SMC01-T_AAAGATGAGGCCGAAT,SMC01,Malignant,3271,32.1296,2.6296,1.6416,0.3352,G1/S,1.6976,Cell Cycle - G1/S,...,157,CMS3,NaN,Mutant,Wildtype,Mutant,Mutant,Wildtype,PM-PS-0001-T-A1,PM-PS-0001-N-A1
SMC01-T_AAAGATGTCACGACTA,SMC01,Malignant,2944,30.4858,-0.8578,-0.1578,-0.0523,Not cycling,1.4116,PDAC classical,...,157,CMS3,NaN,Mutant,Wildtype,Mutant,Mutant,Wildtype,PM-PS-0001-T-A1,PM-PS-0001-N-A1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SMC25-T_TTCTCAAAGGTTCCTA,SMC25,Malignant,4334,-4.8251,13.7615,-0.0350,0.0504,Not cycling,0.6714,Translation initiation,...,165,CMS2,CMS2,Wildtype,Wildtype,Mutant,Mutant,Wildtype,PM-PS-0025-T-A1,NaN
SMC25-T_TTGAACGGTATGCTTG,SMC25,Malignant,4697,-5.4929,13.7057,0.2976,0.0221,Not cycling,0.7400,Secreted I,...,165,CMS2,CMS2,Wildtype,Wildtype,Mutant,Mutant,Wildtype,PM-PS-0025-T-A1,NaN
SMC25-T_TTGGAACCAGGGTATG,SMC25,Malignant,3373,-5.5128,13.6830,0.0200,-0.0706,Not cycling,0.8917,PDAC classical,...,165,CMS2,CMS2,Wildtype,Wildtype,Mutant,Mutant,Wildtype,PM-PS-0025-T-A1,NaN


In [27]:
primary_or_metastatic = []
for i in ad.obs['TNM stage']:
    if i.__contains__('M1'):
        primary_or_metastatic.append('Metastatic')
    else:
        primary_or_metastatic.append('Primary')

In [28]:
# add more clinical information
ad.obs['Project_ID'] = 'GSE132465'
ad.obs['Primary_or_Metastatic'] = primary_or_metastatic
ad.obs['Final_cancer_type'] = 'Colorectal Cancer'
ad.obs['Final_histological_subtype'] = 'Unknown'
ad.obs['Final_molecular_subtype'] = 'Unknown'
ad.obs['Final_tissue'] = 'Colon'
ad.obs['Final_sample_id'] = ad.obs['sample']
ad.obs['Final_patient_age'] = ad.obs['Age']
ad.obs['Final_patient_stage'] = ad.obs['Stage']
ad.obs['Final_patient_treatment'] = 'Naive'

In [29]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/GSE132465.COAD.h5ad', compression='gzip')

## Integrate the data

In [32]:
data_dir = '../../Data/Cancer_cell_data_reprocessed/'
all_h5_files = os.listdir(data_dir)
all_h5_files.sort()

all_h5_files

['2098-Colorectalcancer.COAD.h5ad',
 '2102-Breastcancer.BRCA.h5ad',
 'GSE132465.COAD.h5ad',
 'GSE161529.BRCA.h5ad',
 'GSE167036.BRCA.h5ad',
 'GSE178341.COAD.h5ad',
 'GSE225600.BRCA.h5ad',
 'GSE225857.COAD.h5ad',
 'Multi_modal_breast_cancer.BRCA.h5ad',
 'Wu_etal_2021_BRCA.BRCA.h5ad']

In [ ]:
from collections import defaultdict

cancer_ad_list = []

for h5 in all_h5_files:
    if 'ntegrated' in h5 or 'COAD' not in h5:
        continue

    print(h5)
    # continue
    ad = sc.read_h5ad(data_dir + h5)
    
    if h5.__contains__('2098-Colorectalcancer'):
        ad.obs_names = ad.obs['Cell']
    # display(ad.to_df())
    # continue
    # Fix .var_names
    if ad.var_names[0].startswith('ENSG'):
        new_names = [i.split('_')[0] for i in ad.var.feature_name]
    elif 'ENSG' in ad.var_names[0]:
        new_names = [i.split('_')[0] for i in ad.var_names]
    else:
        new_names = list(ad.var_names)

    # Assign new names
    ad.var_names = new_names
    ad.var_names_make_unique()

    # Fix raw.var names
    if ad.raw is not None:
        ad.raw._var.index = pd.Index(new_names).astype(str)
        # Ensure uniqueness
        seen = defaultdict(int)
        unique_names = []
        for name in ad.raw._var.index:
            if seen[name]:
                unique_names.append(f"{name}_{seen[name]}")
            else:
                unique_names.append(name)
            seen[name] += 1
        ad.raw._var.index = pd.Index(unique_names)

    # Clean obs + var
    # ad.obs = ad.obs.reset_index(drop=True)
    ad.obs_names_make_unique()
    ad.var_names_make_unique()

    cancer_ad_list.append(ad)
    display(ad.to_df())
    display(ad.raw.to_adata().to_df())
    print(ad.raw.to_adata().to_df().max(axis=1))


In [ ]:
memory_usgae()

In [ ]:
for ad in cancer_ad_list:
    print(ad.raw.shape)

In [ ]:
memory_usgae()

In [ ]:
combined_ad = ann.concat(cancer_ad_list, join="inner", axis=0)
combined_ad

In [ ]:
combined_ad.raw.shape

In [ ]:
combined_ad = reprocess_all(combined_ad)

In [ ]:
combined_ad.raw.shape

In [ ]:
combined_ad.obs["Final_histological_subtype"].value_counts()

In [ ]:
combined_ad.obs["Final_histological_subtype_backup"] = combined_ad.obs["Final_histological_subtype"].copy()


In [ ]:
def unify_crc_histology(val):
    val = str(val).lower()
    if "mucinous" in val and "neuroendocrine" in val:
        subtype = "Mucinous neuroendocrine carcinoma"
    elif "mucinous" in val:
        subtype = "Mucinous adenocarcinoma"
    elif "neuroendocrine" in val:
        subtype = "Neuroendocrine tumor"
    elif "medullary" in val:
        subtype = "Medullary carcinoma"
    elif "adenocarcinoma" in val:
        subtype = "Adenocarcinoma"
    elif "unspecified" in val:
        subtype = "Unspecified"
    else:
        subtype = "Unspecified"
    
    return f"COAD: {subtype}"


combined_ad.obs["Final_histological_subtype"] = combined_ad.obs["Final_histological_subtype_backup"].apply(unify_crc_histology)
combined_ad.obs['Final_histological_subtype'].value_counts()

In [ ]:
combined_ad.obs['Final_molecular_subtype'].value_counts()

In [ ]:
combined_ad.obs["Final_molecular_subtype_backup"] = combined_ad.obs["Final_molecular_subtype"].copy()


In [ ]:
def unify_crc_molecular(val):
    val = str(val).strip().lower().replace("coad:", "").strip()
    
    if val in {"mmrp", "mss"}:
        subtype = "MMRp"
    elif val in {"mmrd", "msi-high"}:
        subtype = "MMRd"
    elif "unspecified" in val or 'unknown' in val:
        subtype = "Unspecified"
    else:
        subtype = val.capitalize()
    
    return f"COAD: {subtype}"


combined_ad.obs["Final_molecular_subtype"] = combined_ad.obs["Final_molecular_subtype_backup"].apply(unify_crc_molecular)
combined_ad.obs['Final_molecular_subtype'].value_counts()

In [ ]:
combined_ad.obs["Primary_or_Metastatic"].value_counts()

In [ ]:
def map_primary_metastatic(val):
    val = str(val).strip().lower()
    if val == "m0":
        return "Primary"
    else:
        return val.capitalize()  # keeps values like 'Metastatic', 'Primary', etc.

combined_ad.obs["Primary_or_Metastatic"] = combined_ad.obs["Primary_or_Metastatic"].apply(map_primary_metastatic)


In [ ]:
for obs in ['Project_ID', 'Primary_or_Metastatic', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue']:
    sc.pl.umap(combined_ad, color=obs)

### Harmony integration

In [ ]:
combined_ad

In [ ]:
Z = harmonize(combined_ad.obsm['X_pca'], combined_ad.obs, batch_key = ['Project_ID'])


In [ ]:
combined_ad.obsm['X_pca_harmony'] = Z


In [ ]:
sc.pp.neighbors(combined_ad, n_neighbors=15, use_rep='X_pca_harmony')
sc.tl.umap(combined_ad)

In [ ]:
for obs in ['Project_ID', 'Primary_or_Metastatic', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue']:
    sc.pl.umap(combined_ad, color=obs)

In [ ]:
combined_ad.to_df()

In [ ]:
combined_ad.obs["Final_patient_age_backup"] = combined_ad.obs["Final_patient_age"]

def clean_patient_age(age):
    if pd.isna(age):
        return np.nan
    age = str(age).strip()
    if age.lower() == "unknown":
        return np.nan
    elif "-" in age:
        # Convert age ranges like '46-50' to their midpoint
        parts = age.split("-")
        try:
            return int((int(parts[0]) + int(parts[1])) / 2)
        except:
            return np.nan
    else:
        try:
            return int(age)
        except:
            return np.nan

# Apply cleaning
combined_ad.obs["Final_patient_age"] = combined_ad.obs["Final_patient_age_backup"].apply(clean_patient_age)
combined_ad.obs["Final_patient_age_backup"] =combined_ad.obs["Final_patient_age_backup"].astype(str)

In [ ]:

combined_ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/COAD_integrated.harmony.h5ad', compression='gzip')
